In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, BooleanType,
    LongType, TimestampType, StringType, ArrayType
)
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, lit, desc
import re

spark = SparkSession.builder \
    .appName("traffice-analysis") \
    .master("local[*]") \
    .getOrCreate()

In [46]:

# Read raw TSV — event_list and product_list ingested as StringType first
raw_schema = StructType([
    StructField("hit_time_gmt",   LongType(),      False),
    StructField("date_time",      TimestampType(), False),
    StructField("user_agent",     StringType(),    True),
    StructField("ip",             StringType(),    True),
    StructField("event_list",     StringType(),    True),
    StructField("geo_city",       StringType(),    True),
    StructField("geo_region",     StringType(),    True),
    StructField("geo_country",    StringType(),    True),
    StructField("pagename",       StringType(),    True),
    StructField("page_url",       StringType(),    True),
    StructField("product_list",   StringType(),    True),
    StructField("referrer",       StringType(),    True),
])

df = (
    spark.read.csv(
        "./data/data.sql",
        schema=raw_schema,
        sep="\t",
        header=True,
        timestampFormat="yyyy-MM-dd HH:mm:ss",
    )
    # event_list: comma-separated event codes → Array[String]
    .withColumn(
        "event_list",
        F.when(F.col("event_list").isNotNull(),
               F.split(F.col("event_list"), ",").cast(ArrayType(IntegerType())))
         .otherwise(F.lit(None))
    )
    # product_list: comma-separated product entries → Array[String]
    .withColumn(
        "product_list",
        F.when(F.col("product_list").isNotNull(),
               F.split(F.col("product_list"), ","))
         .otherwise(F.lit(None))
    )
)

df.printSchema()

root
 |-- hit_time_gmt: long (nullable = true)
 |-- date_time: timestamp (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- ip: string (nullable = true)
 |-- event_list: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- geo_city: string (nullable = true)
 |-- geo_region: string (nullable = true)
 |-- geo_country: string (nullable = true)
 |-- pagename: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- referrer: string (nullable = true)



In [47]:
df.select("hit_time_gmt", "referrer", "product_list", "event_list").show()

+------------+--------------------+--------------------+----------+
|hit_time_gmt|            referrer|        product_list|event_list|
+------------+--------------------+--------------------+----------+
|  1254033280|http://www.google...|                NULL|      NULL|
|  1254033379|http://www.bing.c...|[Electronics;Zune...|       [2]|
|  1254033478|http://search.yah...|                NULL|      NULL|
|  1254033577|http://www.google...|                NULL|      NULL|
|  1254033676|http://www.esshop...|                NULL|      NULL|
|  1254033775|http://www.esshop...|                NULL|      [12]|
|  1254033874|http://www.esshop...|[Electronics;Ipod...|       [2]|
|  1254033973|http://www.esshop...|[Electronics;Ipod...|       [2]|
|  1254034072|http://www.esshop...|                NULL|      [11]|
|  1254034171|http://www.esshop...|                NULL|      [12]|
|  1254034270|http://www.esshop...|                NULL|      NULL|
|  1254034369|https://www.essho...|             

In [48]:

def extract_domain(url):
    if url is None:
        return None
    # Remove protocol (http://, https://, ftp://, etc.)
    domain = re.sub(r'^[a-zA-Z][a-zA-Z0-9+\-.]*://', '', url)
    # Remove everything after the first slash (path, query, fragment)
    domain = domain.split('/')[0]
    # Remove port number if present
    domain = domain.split(':')[0]
    # Remove 'www.' prefix if present
    domain = re.sub(r'^www\.', '', domain)
    return domain if domain else None

extract_domain_udf = udf(extract_domain, StringType())

In [49]:
df.select(extract_domain_udf("referrer")).show()

+------------------------+
|extract_domain(referrer)|
+------------------------+
|              google.com|
|                bing.com|
|        search.yahoo.com|
|              google.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
|         esshopzilla.com|
+------------------------+
only showing top 20 rows


In [125]:
def get_rev(product_list):
    total = 0

    if not product_list:
        return total
    
    try:
        # check if product_list is type list
        if isinstance(product_list, list) and len(product_list) > 0:
            # Iterate through product_list
            for product in product_list:
                # Get Total revenu is index 3
                p = product.split(";")[3]          
                # Check the value and evluate to zero if not
                p_rev = 0 if not p else int(p)
                # print(f"p_rev: {p_rev}")
                total = total + p_rev
        else:
            # Total revenu is index 3
            p = product.split(";")[3]             
            total = 0 if not p else int(p) 
    except Exception as e:
        print(f"Error get_rev function: {e}")

    # print(f"total: {total}")
    return total

get_rev_udf = udf(get_rev, IntegerType())

In [51]:
df.select("event_list", "referrer", get_rev_udf("product_list")).show(truncate=False)

+----------+------------------------------------------------------------------------------------------------------------------------+---------------------+
|event_list|referrer                                                                                                                |get_rev(product_list)|
+----------+------------------------------------------------------------------------------------------------------------------------+---------------------+
|NULL      |http://www.google.com/search?hl=en&client=firefox-a&rls=org.mozilla%3Aen-US%3Aofficial&hs=ZzP&q=Ipod&aq=f&oq=&aqi=      |0                    |
|[2]       |http://www.bing.com/search?q=Zune&go=&form=QBLH&qs=n                                                                    |0                    |
|NULL      |http://search.yahoo.com/search?p=cd+player&toggle=1&cop=mss&ei=UTF-8&fr=yfp-t-701                                       |0                    |
|NULL      |http://www.google.com/search?hl=en&client=firefox-a&

p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [55]:
def is_purchase_order(event_list: list[int], purchase_event: int):
    if not event_list:
        return False
    
    return True if purchase_event in event_list else False

is_purchase_order_udf = udf(is_purchase_order, BooleanType())

In [56]:
print(is_purchase_order([1, 1, 3, 4], 2))

False


In [93]:
# df.createOrReplaceTempView("traffic_data")

In [89]:
from pyspark.sql.functions import col, sum as spark_sum

df = df.withColumn("domain", extract_domain_udf("referrer")).withColumn("revenue", get_rev_udf("product_list"))
# df.show()

purchased_traffice = (
    df
    .select("ip", "revenue")
    .filter(is_purchase_order_udf(col("event_list"), lit(1)))
    .groupBy("ip")
    .agg(spark_sum("revenue").alias("total_revenue"))
)
purchased_traffice.show(truncate=False)

+-----------+-------------+
|ip         |total_revenue|
+-----------+-------------+
|44.12.96.2 |190          |
|23.8.61.21 |250          |
|67.98.123.1|290          |
+-----------+-------------+



p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [116]:
import re
from urllib.parse import urlparse, parse_qs, unquote

def extract_keywords(url: str):
    try:
        parsed_url = urlparse(url)
        query_params = parse_qs(parsed_url.query)
        keywords_list = []
        
        # Define common keys that indicate search keywords
        keyword_keys = {'q', 'p'}
        
        for key, values in query_params.items():
            if key.lower() in keyword_keys:
                for value in values:
                    decoded_value = unquote(value)
                    clean_value = decoded_value.strip()
                    if clean_value:
                        keywords_list.append(clean_value)
        
        final_keywords = ", ".join(keywords_list)
    except Excpeption as e:
        print(f"Error extract_keywords func")
        return None

    return final_keywords

extract_keywords_udf = udf(extract_keywords, StringType())

In [120]:
df.withColumn('keywords', extract_keywords_udf("referrer")).show()

+------------+-------------------+--------------------+-------------+----------+--------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+----------------+-------+---------+
|hit_time_gmt|          date_time|          user_agent|           ip|event_list|      geo_city|geo_region|geo_country|            pagename|            page_url|        product_list|            referrer|          domain|revenue| keywords|
+------------+-------------------+--------------------+-------------+----------+--------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+----------------+-------+---------+
|  1254033280|2009-09-27 06:34:40|Mozilla/5.0 (Wind...|  67.98.123.1|      NULL|         Salem|        OR|         US|                Home|http://www.esshop...|                NULL|http://www.google...|      google.com|      0|     Ipod|
|  1254033379|2009-09-27 06:36:19|Mozilla/5.0 (M

p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 0
total: 0
p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [121]:
df.select("keywords").show()

{"ts": "2026-04-23 22:47:27.761", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `keywords` cannot be resolved. Did you mean one of the following? [`domain`, `ip`, `referrer`, `revenue`, `geo_city`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor30.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1278.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `keywords` cannot be resolved. Did you mean one of the following? [`domain`, `ip`, `referrer`, `revenue`, `geo_city`]. SQLSTATE: 42703;\n'Project ['keywords]\n+- Project [hit_time_gmt#465L, date_time#466, user_agent#467, ip#468, event_list#478, geo_city#470, geo_region#471

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `keywords` cannot be resolved. Did you mean one of the following? [`domain`, `ip`, `referrer`, `revenue`, `geo_city`]. SQLSTATE: 42703;
'Project ['keywords]
+- Project [hit_time_gmt#465L, date_time#466, user_agent#467, ip#468, event_list#478, geo_city#470, geo_region#471, geo_country#472, pagename#473, page_url#474, product_list#479, referrer#476, domain#1195, get_rev(product_list#479)#1196 AS revenue#1197]
   +- Project [hit_time_gmt#465L, date_time#466, user_agent#467, ip#468, event_list#478, geo_city#470, geo_region#471, geo_country#472, pagename#473, page_url#474, product_list#479, referrer#476, extract_domain(referrer#476)#1194 AS domain#1195, revenue#1166]
      +- Project [hit_time_gmt#465L, date_time#466, user_agent#467, ip#468, event_list#478, geo_city#470, geo_region#471, geo_country#472, pagename#473, page_url#474, product_list#479, referrer#476, domain#1164, get_rev(pr

In [123]:

# Visit traffic
# df.show()
visit_traffic = df.select("ip", "domain", "referrer").filter((col("domain") != "esshopzilla.com")).withColumn('keywords', extract_keywords_udf("referrer"))
# visit_traffic = df.select("ip", "referrer")
visit_traffic.show(truncate=False)

+-------------+----------------+------------------------------------------------------------------------------------------------------------------------+---------+
|ip           |domain          |referrer                                                                                                                |keywords |
+-------------+----------------+------------------------------------------------------------------------------------------------------------------------+---------+
|67.98.123.1  |google.com      |http://www.google.com/search?hl=en&client=firefox-a&rls=org.mozilla%3Aen-US%3Aofficial&hs=ZzP&q=Ipod&aq=f&oq=&aqi=      |Ipod     |
|23.8.61.21   |bing.com        |http://www.bing.com/search?q=Zune&go=&form=QBLH&qs=n                                                                    |Zune     |
|112.33.98.231|search.yahoo.com|http://search.yahoo.com/search?p=cd+player&toggle=1&cop=mss&ei=UTF-8&fr=yfp-t-701                                       |cd player|
|44.12.96.2   |g

In [124]:
joined = purchased_traffice.join(visit_traffic, on="ip", how="left")
joined.show()

+-----------+-------------+----------+--------------------+--------+
|         ip|total_revenue|    domain|            referrer|keywords|
+-----------+-------------+----------+--------------------+--------+
| 44.12.96.2|          190|google.com|http://www.google...|    ipod|
| 23.8.61.21|          250|  bing.com|http://www.bing.c...|    Zune|
|67.98.123.1|          290|google.com|http://www.google...|    Ipod|
+-----------+-------------+----------+--------------------+--------+



p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [128]:
output = joined.select("domain", "keywords", "total_revenue").sort(desc("total_revenue"))
output.show()

+----------+--------+-------------+
|    domain|keywords|total_revenue|
+----------+--------+-------------+
|google.com|    Ipod|          290|
|  bing.com|    Zune|          250|
|google.com|    ipod|          190|
+----------+--------+-------------+



p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [132]:
from datetime import datetime
today = datetime.now()
output_filename = f"{today.strftime("%Y-%m-%d")}_SearchKeywordPerformance.tab"
print(output_filename)

2026-04-23_SearchKeywordPerformance.tab


In [137]:
output.coalesce(1).write.mode("overwrite").option("header", "true").option("sep", "\t").csv(f"./output/{output_filename}")

p_rev: 250
total: 250
p_rev: 190
total: 190
p_rev: 290
total: 290


In [ ]:
l = ['Computers;HP Pavillion;1;1000;200|201,Office Supplies;Red Folders;4;4.00;205|206|207', 'Computers;HP Pavillion;1;3000;200|201,Office Supplies;Red Folders;4;4.00;205|206|207']
ll = [1, 2, 3]
# print(type(l) is list)
# print(len(l) > 0)

# print(type(l))
print(get_rev(l))